In [1]:
import os, glob, shutil, subprocess, sys, json, time, zipfile
from pathlib import Path
from statistics import mean

WORK_DIR = Path("/kaggle/working/PA2")
os.environ["PYTHONUNBUFFERED"] = "1"
N_EXPECTED = 300
DECODE = "--max_new_tokens 384 --repetition_penalty 1.1 --no_repeat_ngram_size 0"   # identical to the base run

def run(cmd, env_extra=None):
    print("$", cmd)
    env = {**os.environ, **(env_extra or {})}
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    for line in p.stdout:
        print(line, end="")
    rc = p.wait()
    if rc != 0:
        raise RuntimeError(f"command failed with exit code {rc}: {cmd}")

def n_rows(path):
    try:
        return len(json.load(open(path)))
    except Exception:
        return 0


In [2]:
def find_root():
    hits = glob.glob("/kaggle/input/**/scripts/generate.py", recursive=True)
    roots = [Path(h).parent.parent for h in hits]
    good = [r for r in roots if (r / "scripts/make_pairs.py").exists() and (r / "eval_sets").exists()]
    return (good or [None])[0]

root = find_root()
assert root is not None, "No project folder with scripts/generate.py and scripts/make_pairs.py found under /kaggle/input."
print("Project root:", root)
if not (WORK_DIR / "scripts/generate.py").exists():
    shutil.copytree(root, WORK_DIR, dirs_exist_ok=True,
                    ignore=shutil.ignore_patterns("outputs", "__pycache__", "*.egg-info", "*.safetensors"))
os.chdir(WORK_DIR)
sys.path.insert(0, str(WORK_DIR))
for d in ("outputs/gen", "outputs/eval"):
    Path(d).mkdir(parents=True, exist_ok=True)

def find_ckpt(run_dir):
    hits = sorted(Path(h).parent for h in glob.glob(f"/kaggle/input/**/{run_dir}/best/config.json", recursive=True))
    if len(hits) > 1:
        print(f"note: {len(hits)} copies of {run_dir}/best found, using the first:", *hits, sep="\n   ")
    return hits[0] if hits else None

SFT, HH_BEST, UF_BEST = find_ckpt("sft"), find_ckpt("dpo_hh"), find_ckpt("dpo_uf")
ALPACA = "eval_sets/alpaca_eval_300_seed42.json"
HH_EVAL = "eval_sets/hh_rlhf_eval_300_seed42.json"

# base generations (only used for the comparison table in Step 5) and the AlpacaEval subset, if stored elsewhere in the dataset
for name, dst in (("base_alpaca.json", "outputs/gen"), ("alpaca_eval_300_seed42.json", "eval_sets")):
    if not (WORK_DIR / dst / name).exists():
        hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
        if hits:
            shutil.copy(hits[0], WORK_DIR / dst / name)

for name, p in (("SFT best", SFT), ("DPO-HH best", HH_BEST), ("DPO-UF best", UF_BEST)):
    print(f"{name:12s}", p, "| OK" if p and (Path(p) / "model.safetensors").exists() else "| MISSING")
for p in (ALPACA, HH_EVAL, "outputs/gen/base_alpaca.json"):
    print(f"{p:45s}", "OK" if Path(p).exists() else "MISSING (regenerated / skipped below)" if "alpaca" in p else "MISSING")
assert SFT and HH_BEST and UF_BEST, "A checkpoint folder is missing from the attached dataset (see the lines above)."
assert Path(HH_EVAL).exists(), "missing eval_sets/hh_rlhf_eval_300_seed42.json"


Project root: /kaggle/input/datasets/areeshayaseen/cs5326-pa2-gen/PA2
note: 2 copies of sft/best found, using the first:
   /kaggle/input/datasets/areeshayaseen/cs5326-pa2-gen/PA2/outputs/sft/best
   /kaggle/input/notebooks/areeshayaseen/pa2-dpo/PA2/outputs/sft/best
SFT best     /kaggle/input/datasets/areeshayaseen/cs5326-pa2-gen/PA2/outputs/sft/best | OK
DPO-HH best  /kaggle/input/notebooks/areeshayaseen/pa2-dpo/PA2/outputs/dpo_hh/best | OK
DPO-UF best  /kaggle/input/notebooks/areeshayaseen/pa2-dpo/PA2/outputs/dpo_uf/best | OK
eval_sets/alpaca_eval_300_seed42.json         OK
eval_sets/hh_rlhf_eval_300_seed42.json        OK
outputs/gen/base_alpaca.json                  OK


**Install Dependencies**

In [3]:
%cd /kaggle/working/PA2
!python -m pip uninstall -y -q torchao || true
!python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors matplotlib
!python -m pip install -q -e . --no-deps
!nvidia-smi -L
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__, "| GPUs:", torch.cuda.device_count())
assert torch.cuda.device_count() >= 2, "Select GPU T4 x2."

# the AlpacaEval subset is deterministic (seed 42): regenerate it if it was not in the dataset
if not Path(ALPACA).exists():
    run("python scripts/gen_alpaca_subset.py")
assert Path(ALPACA).exists(), "AlpacaEval subset missing"


/kaggle/working
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 95.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 79.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for small-model-alignment-starter (py

**Sanity Check:**

In [4]:
%cd /kaggle/working/PA2
src = open("scripts/generate.py").read()
print("generate.py mentions format_prompt:", "format_prompt" in src, "| TEMPLATE:", "TEMPLATE" in src)
!grep -n "format_prompt\|TEMPLATE\|Instruction\|apply_chat_template" scripts/generate.py
assert "format_prompt" in src or "### Instruction" in src, \
    "generate.py does not appear to use the training prompt template - fix it before generating."

run(f"python scripts/generate.py --model_path {UF_BEST} --eval_path {ALPACA} --out_path outputs/gen/dpo_uf_sanity.json "
    f"--generator dpo-ultrafeedback --limit 3 {DECODE}", {"CUDA_VISIBLE_DEVICES": "0"})
for r in json.load(open("outputs/gen/dpo_uf_sanity.json")):
    print("\nINSTRUCTION:", r["instruction"][:150]); print("OUTPUT     :", r["output"][:400])

# prompt-length check: generate.py truncates prompts to 256 tokens, which would cut off "### Response:" for longer ones
from transformers import AutoTokenizer
from pa2.data import format_prompt
tk = AutoTokenizer.from_pretrained(str(SFT))
for nm, path in (("AlpacaEval", ALPACA), ("HH eval", HH_EVAL)):
    ds = json.load(open(path))
    lens = [len(tk.encode(format_prompt(d["instruction"]), add_special_tokens=False)) for d in ds]
    print(f"{nm}: {len(ds)} prompts | longest {max(lens)} tokens | over 256 (truncated by generate.py): {sum(l > 256 for l in lens)}")


/kaggle/working
generate.py mentions format_prompt: True | TEMPLATE: False
4:from pa2.data import format_prompt
34:    prompts = [format_prompt(d["instruction"]) for d in batch]
$ python scripts/generate.py --model_path /kaggle/input/notebooks/areeshayaseen/pa2-dpo/PA2/outputs/dpo_uf/best --eval_path eval_sets/alpaca_eval_300_seed42.json --out_path outputs/gen/dpo_uf_sanity.json --generator dpo-ultrafeedback --limit 3 --max_new_tokens 384 --repetition_penalty 1.1 --no_repeat_ngram_size 0
The tokenizer you are loading from '/kaggle/input/notebooks/areeshayaseen/pa2-dpo/PA2/outputs/dpo_uf/best' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
`torch_dtype` is deprecated! Use `dtype` instead!
3/3 done

INSTRUCTION: How do I dice without slicing my finger
OUTPUT

The tokenizer you are loading from '/kaggle/input/datasets/areeshayaseen/cs5326-pa2-gen/PA2/outputs/sft/best' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


AlpacaEval: 300 prompts | longest 397 tokens | over 256 (truncated by generate.py): 4
HH eval: 300 prompts | longest 32 tokens | over 256 (truncated by generate.py): 0


**Generation**

In [5]:
%cd /kaggle/working/PA2

def gen_cmd(name, model_path, eval_path, generator, extra=""):
    out = f"outputs/gen/{name}.json"
    if n_rows(out) == N_EXPECTED:
        return None
    return (f"python scripts/generate.py --model_path {model_path} --eval_path {eval_path} --out_path {out} "
            f"--generator {generator} {extra} {DECODE}")

chains = {
    "0": [gen_cmd("sft_alpaca", SFT, ALPACA, "sft"),
          gen_cmd("dpo_uf_alpaca", UF_BEST, ALPACA, "dpo-ultrafeedback")],
    "1": [gen_cmd("sft_hh", SFT, HH_EVAL, "sft", "--dataset hh_eval"),
          gen_cmd("dpo_hh_hh", HH_BEST, HH_EVAL, "dpo-hh", "--dataset hh_eval")],
}
procs, logs = {}, {}
for gpu, cmds in chains.items():
    cmds = [c for c in cmds if c]
    if not cmds:
        print(f"GPU {gpu}: nothing to do")
        continue
    logs[gpu] = f"/kaggle/working/gen_gpu{gpu}.log"
    procs[gpu] = subprocess.Popen(" && ".join(cmds), shell=True, stdout=open(logs[gpu], "w"),
                                  stderr=subprocess.STDOUT, env={**os.environ, "CUDA_VISIBLE_DEVICES": gpu})
    print(f"GPU {gpu}: started {len(cmds)} job(s)")

def last_line(path):
    try:
        txt = open(path, errors="ignore").read().replace("\r", "\n")
        lines = [l for l in txt.splitlines() if l.strip()]
        return lines[-1][:200] if lines else ""
    except Exception:
        return ""

t0 = time.time()
while procs and any(p.poll() is None for p in procs.values()):
    time.sleep(120)
    for gpu, p in procs.items():
        state = "running" if p.poll() is None else f"exit {p.returncode}"
        print(f"[{(time.time()-t0)/60:5.1f} min] GPU {gpu} {state}: {last_line(logs[gpu])}")
for gpu, p in procs.items():
    if p.returncode != 0:
        print(open(logs[gpu], errors="ignore").read()[-3000:])
        raise RuntimeError(f"generation on GPU {gpu} failed (exit {p.returncode}); see log above")
print(f"All generation finished in {(time.time()-t0)/60:.1f} min")


/kaggle/working
GPU 0: started 2 job(s)
GPU 1: started 2 job(s)
[  2.0 min] GPU 0 running: 96/300 done
[  2.0 min] GPU 1 running: 96/300 done
[  4.0 min] GPU 0 running: 272/300 done
[  4.0 min] GPU 1 running: 288/300 done
[  6.0 min] GPU 0 running: 96/300 done
[  6.0 min] GPU 1 running: 64/300 done
[  8.0 min] GPU 0 running: 240/300 done
[  8.0 min] GPU 1 running: 256/300 done
[ 10.0 min] GPU 0 exit 0: 300/300 done
[ 10.0 min] GPU 1 exit 0: 300/300 done
All generation finished in 10.0 min


**Quality Check**

In [6]:
%cd /kaggle/working/PA2

def rep4(t):
    w = t.split(); g = [tuple(w[i:i+4]) for i in range(len(w)-3)]
    return 0.0 if not g else 1 - len(set(g)) / len(g)

files = ["base_alpaca", "sft_alpaca", "dpo_uf_alpaca", "sft_hh", "dpo_hh_hh"]
print(f"{'file':16s} {'rows':>5s} {'empty':>6s} {'words(mean)':>12s} {'no end punct':>13s} {'rep4':>6s}")
data = {}
for f in files:
    p = Path(f"outputs/gen/{f}.json")
    if not p.exists():
        print(f"{f:16s} MISSING"); continue
    rows = json.load(open(p)); data[f] = rows
    outs = [r.get("output", "") for r in rows]
    print(f"{f:16s} {len(rows):5d} {sum(1 for o in outs if not o.strip()):6d} {mean(len(o.split()) for o in outs):12.0f} "
          f"{sum(1 for o in outs if o.strip() and o.strip()[-1] not in '.!?)\"`*'):13d} {mean(map(rep4, outs)):6.3f}")

for a, b in (("sft_alpaca", "dpo_uf_alpaca"), ("sft_hh", "dpo_hh_hh")):
    if a in data and b in data:
        same = [r["id"] for r in data[a]] == [r["id"] for r in data[b]]
        print(f"{a} vs {b}: same prompt ids and order = {same}")
        r1, r2 = data[a][5], data[b][5]
        print("\nPROMPT:", r1["instruction"][:200]); print("SFT:", r1["output"][:400].replace("\n", " ")); print("DPO:", r2["output"][:400].replace("\n", " "))


/kaggle/working
file              rows  empty  words(mean)  no end punct   rep4
base_alpaca        300      0          181            76  0.032
sft_alpaca         300      0          124            47  0.031
dpo_uf_alpaca      300      0          173            67  0.032
sft_hh             300      0          133            33  0.013
dpo_hh_hh          300      0          123            25  0.016
sft_alpaca vs dpo_uf_alpaca: same prompt ids and order = True

PROMPT: Are there any weird Christmas traditions?
SFT: Yes, there are many weird Christmas traditions around the world. Here are a few examples:  1. The "Christmas Tree" tradition: In some cultures, people decorate their homes with a tree that is decorated with ornaments and other decorations.  2. The "Halloween" tradition: Halloween involves trick-or-treating, costumes, candy, and parties.  3. The "Christmas Eve" tradition: This is a day of rest and 
DPO: Yes, there are many weird Christmas traditions around the world. Here are a 

**Randomized pairs**

In [7]:
%cd /kaggle/working/PA2
run("python scripts/make_pairs.py --sft_path outputs/gen/sft_alpaca.json --dpo_path outputs/gen/dpo_uf_alpaca.json --out_path outputs/eval/uf_pairs.jsonl")
run("python scripts/make_pairs.py --sft_path outputs/gen/sft_hh.json --dpo_path outputs/gen/dpo_hh_hh.json --out_path outputs/eval/hh_pairs.jsonl")

for f in ("outputs/eval/uf_pairs.jsonl", "outputs/eval/hh_pairs.jsonl"):
    rows = [json.loads(l) for l in open(f) if l.strip()]
    print(f"\n{f}: {len(rows)} pairs | keys: {list(rows[0].keys())}")
    print("DPO in position A:", sum(1 for r in rows if r["meta"]["a_is"] == "dpo"), "of", len(rows), "(should be near half)")


/kaggle/working
$ python scripts/make_pairs.py --sft_path outputs/gen/sft_alpaca.json --dpo_path outputs/gen/dpo_uf_alpaca.json --out_path outputs/eval/uf_pairs.jsonl
wrote outputs/eval/uf_pairs.jsonl
$ python scripts/make_pairs.py --sft_path outputs/gen/sft_hh.json --dpo_path outputs/gen/dpo_hh_hh.json --out_path outputs/eval/hh_pairs.jsonl
wrote outputs/eval/hh_pairs.jsonl

outputs/eval/uf_pairs.jsonl: 300 pairs | keys: ['id', 'instruction', 'response_a', 'response_b', 'meta']
DPO in position A: 151 of 300 (should be near half)

outputs/eval/hh_pairs.jsonl: 300 pairs | keys: ['id', 'instruction', 'response_a', 'response_b', 'meta']
DPO in position A: 151 of 300 (should be near half)


**Package files**

In [8]:
%cd /kaggle/working/PA2
zpath = "/kaggle/working/PA2_gen_outputs.zip"
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as zf:
    for pattern in ("outputs/gen/*.json", "outputs/eval/*.jsonl"):
        for f in glob.glob(pattern):
            zf.write(f, arcname="PA2/" + f)
print("Created", zpath, f"({os.path.getsize(zpath)/1e6:.2f} MB)")
!ls -la outputs/gen outputs/eval


/kaggle/working
Created /kaggle/working/PA2_gen_outputs.zip (0.86 MB)
outputs/eval:
total 1156
drwxr-xr-x 2 root root   4096 Oct 13 12:27 .
drwxr-xr-x 4 root root   4096 Oct 13 12:15 ..
-rw-r--r-- 1 root root 527840 Oct 13 12:27 hh_pairs.jsonl
-rw-r--r-- 1 root root 646151 Oct 13 12:27 uf_pairs.jsonl

outputs/gen:
total 1760
drwxr-xr-x 2 root root   4096 Oct 13 12:22 .
drwxr-xr-x 4 root root   4096 Oct 13 12:15 ..
-rw-r--r-- 1 root root 442883 Oct 13 12:15 base_alpaca.json
-rw-r--r-- 1 root root 286226 Oct 13 12:25 dpo_hh_hh.json
-rw-r--r-- 1 root root 422145 Oct 13 12:26 dpo_uf_alpaca.json
-rw-r--r-- 1 root root   4242 Oct 13 12:17 dpo_uf_sanity.json
-rw-r--r-- 1 root root 318457 Oct 13 12:21 sft_alpaca.json
-rw-r--r-- 1 root root 305509 Oct 13 12:21 sft_hh.json
